# ABSA Model Training, Saving & Inference

Uses `model.model.ABSAModel` (multi-head softmax architecture) and  
`model.train.train_model` (HF Trainer with weighted cross-entropy).

In [ ]:
import os
import torch
import pandas as pd
from torch.utils.data import random_split
from transformers import AutoTokenizer, set_seed

from config.global_config import SENTIMENT_LABELS, TRAIN_ASPECTS
from model.model import ABSAModel
from model.prepare_dataset import ABSADatasetMultiHead
from model.train import train_model, compute_class_weights

MODEL_NAME = "distilbert-base-uncased"
DATA_PATH = "statics/datasets/training.csv"
SAVE_DIR = "saved_models"
VAL_RATIO = 0.2
SEED = 25

set_seed(SEED)
print(f"Model:      {MODEL_NAME}")
print(f"Aspects:    {TRAIN_ASPECTS}")
print(f"Sentiments: {SENTIMENT_LABELS}")

In [ ]:
df = pd.read_csv(DATA_PATH)
df[TRAIN_ASPECTS] = df[TRAIN_ASPECTS].fillna("notmentioned")
print(f"Loaded {len(df)} examples")

for aspect in TRAIN_ASPECTS:
    print(f"  {aspect}: {dict(df[aspect].value_counts())}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
dataset = ABSADatasetMultiHead(df, tokenizer)
labels_np = dataset.get_labels_numpy()

val_size = max(1, int(len(dataset) * VAL_RATIO))
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

In [ ]:
class_weights = compute_class_weights(labels_np)
print(f"Class weights: {class_weights}")

model = ABSAModel(MODEL_NAME, class_weights=class_weights)
print(f"Total params:     {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

trainer = train_model(
    model=model,
    train_dataset=train_ds,
    val_dataset=val_ds,
    output_dir="./results",
)

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)

safe_name = MODEL_NAME.replace("/", "_")
save_path = os.path.join(SAVE_DIR, f"{safe_name}_absa.pt")

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "base_model_name": MODEL_NAME,
    },
    save_path,
)
print(f"Model saved → {save_path}")

---
## Load the saved model & run inference

In [ ]:
from model.model import ABSAModel
from model.predict import predict
from transformers import AutoTokenizer

checkpoint = torch.load(save_path, weights_only=False)
base_model_name = checkpoint["base_model_name"]
print(f"Rebuilding model from backbone: {base_model_name}")

loaded_model = ABSAModel(base_model_name)
loaded_model.load_state_dict(checkpoint["model_state_dict"], strict=False)
loaded_model.eval()

loaded_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
print("Model loaded successfully!")

In [ ]:
test_texts = [
    "This park is dirty and unsafe at night, the paths are broken.",
    "Beautiful museum with free entry and amazing historical exhibitions!",
]

for text in test_texts:
    results, probabilities = predict(text, loaded_model, loaded_tokenizer)

    print(f"\n>>> {text}")
    active = {k: v for k, v in results.items() if v != "notmentioned"}
    print(f"  Active sentiments: {active if active else 'None'}")
    print(f"  Full predictions:  {results}")